# Saudi Compliance AI — RAG Development

Clean notebook for the current local RTX 5080 + Gemma 3 12B setup.

**Before running:** start Gemma from Command Prompt with the server command we already used. This notebook does not start or stop Gemma.


In [1]:
# 1. Imports
import json
import re
import numpy as np
import faiss
import requests
from pathlib import Path
from sentence_transformers import SentenceTransformer


In [2]:
# 2. Project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)


Project root: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI
Processed directory: C:\Users\User\OneDrive\Desktop\Saudi-Compliance-AI\data\processed


In [3]:
# 3. Load the current RAG files
with open(PROCESSED_DIR / "rag_chunks.json", "r", encoding="utf-8") as f:
    rag_chunks = json.load(f)

rag_embeddings = np.load(PROCESSED_DIR / "rag_embeddings.npy")
faiss_index = faiss.read_index(str(PROCESSED_DIR / "faiss.index"))

print("Chunks:", len(rag_chunks))
print("Embeddings:", rag_embeddings.shape)
print("FAISS vectors:", faiss_index.ntotal)
print("FAISS dimension:", faiss_index.d)


Chunks: 162
Embeddings: (162, 768)
FAISS vectors: 162
FAISS dimension: 768


In [4]:
# 4. Arabic normalization
def normalize_arabic(text):
    text = text.lower()
    text = text.replace("ـ", "")
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")
    text = text.replace("ى", "ي")
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# This derived field is recreated every time the notebook opens.
for chunk in rag_chunks:
    chunk["normalized_text"] = normalize_arabic(chunk["text"])

print("Arabic normalization ready.")


Arabic normalization ready.


In [5]:
# 5. Load the embedding model on the RTX 5080
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
    device="cuda"
)

print("Embedding model loaded.")
print("Device:", embedding_model.device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.
Device: cuda:0


In [6]:
# 6. Hybrid retrieval
def hybrid_search(question, k=5):
    normalized_question = normalize_arabic(question)

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    semantic_scores, indices = faiss_index.search(
        np.array(query_embedding, dtype="float32"),
        len(rag_chunks)
    )

    semantic_score_map = {
        int(idx): float(score)
        for idx, score in zip(indices[0], semantic_scores[0])
    }

    query_words = set(
        word for word in normalized_question.split()
        if len(word) >= 3
    )

    results = []

    for idx, chunk in enumerate(rag_chunks):
        normalized_text = chunk["normalized_text"]

        matched_words = [
            word for word in query_words
            if word in normalized_text
        ]

        keyword_score = (
            len(matched_words) / len(query_words)
            if query_words else 0.0
        )

        semantic_score = semantic_score_map.get(idx, 0.0)

        hybrid_score = (
            0.7 * semantic_score
            + 0.3 * keyword_score
        )

        results.append({
            "idx": idx,
            "semantic_score": semantic_score,
            "keyword_score": keyword_score,
            "hybrid_score": hybrid_score,
            "matched_words": matched_words
        })

    results.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    return results[:k]


In [7]:
# 7. Quick retrieval check
query = "كم عدد ساعات العمل الإضافية المسموح بها خلال السنة؟"
results = hybrid_search(query, k=3)

print("Question:", query)
print()

for rank, result in enumerate(results, start=1):
    chunk = rag_chunks[result["idx"]]
    metadata = chunk["metadata"]

    print(f"--- Rank {rank} ---")
    print("Hybrid score:", round(result["hybrid_score"], 4))
    print("Type:", metadata["article_type"])
    print("Article:", metadata["article"])
    print("Page:", metadata["page_start"])
    print("Matched words:", result["matched_words"])
    print()
    print(chunk["text"])
    print()


Question: كم عدد ساعات العمل الإضافية المسموح بها خلال السنة؟

--- Rank 1 ---
Hybrid score: 0.7004
Type: executive_regulation
Article: 22
Page: 17
Matched words: ['خلال', 'ساعات', 'الاضافية', 'عدد', 'العمل']

في تنفيذ أحكام )المادة السـادسـة بعد المائة( من النظام لا يجوز أن تزيد سـاعات العمل الإضـافية خلال السـنة على سبعمائة وعشرين ساعة؛
ويجوز بموافقة العامل زيادة عدد الساعات الإضافية عن ذلك.

--- Rank 2 ---
Hybrid score: 0.5846
Type: labor_law
Article: السابعة بعد المائة
Page: 17
Matched words: ['ساعات', 'الاضافية', 'العمل']

1. يجب على صاحب العمل أن يدفع للعامل أجراً إضافيًّا عن ساعات العمل الإضافية يوازي أجر الساعة مضافًا إليه )50
%
( من أجره
الأساسي، ويجوز لصاحب العمل بموافقة العامل أن يحتسب للعامل أيام إجازة تعويضية مدفوعة الأجر بدلاً عن الأجر المستحق للعامل
لساعات العمل الإضافية، وتبين اللائحة الأحكام المتصلة بذلك.
2. إذا كان التشغيل في المنشأة على أساس المعيار الأسبوعي لساعات العمل تعد الساعات التي تزيد على الساعات المتخذة لهذا
المعيار ساعات عمل إضافية.
3. تعد جميع ساعات العمل ا

## 8. Gemma instructions

Gemma must answer only from the retrieved legal context and must distinguish the source types.


In [8]:
# 8. System prompt
system_prompt = """أنت مساعد متخصص في نظام العمل السعودي واللوائح التنفيذية المرتبطة به.

قواعد الإجابة الصارمة:

1. اعتمد فقط على النصوص القانونية الموجودة في السياق المسترجع.
2. لا تستخدم أي معلومة من معرفتك العامة خارج السياق.
3. أجب مباشرة عن سؤال المستخدم وباختصار.
4. لا تخلط بين الأحكام الواردة في مصادر مختلفة بطريقة تغيّر معناها.
5. إذا كانت هناك عدة مواد مرتبطة بالسؤال، اشرح حكم كل مادة بشكل منفصل إذا كان ذلك ضروريًا لفهم الإجابة.
6. لا تستنتج وجود حق أو التزام أو مدة أو استثناء إلا إذا كان النص المسترجع يدعمه بوضوح.
7. لا تعتبر حكمين متشابهين حكمًا واحدًا، ولا تضفهما إلى بعضهما إلا إذا نص السياق بوضوح على ذلك.
8. حافظ على أرقام وأسماء المواد كما وردت في السياق.
9. عند ذكر حكم قانوني، اذكر المادة ومصدرها.
10. إذا كان السياق المسترجع لا يحتوي على معلومات كافية للإجابة، قل:
"لا يحتوي السياق المسترجع على معلومات كافية للإجابة."
11. أجب باللغة العربية فقط.
12. لا تستخدم لغة أخرى.
13. لا تكرر النص القانوني حرفيًا؛ لخّص الحكم المطلوب بدقة.
"""

print("System prompt ready.")


System prompt ready.


In [12]:
# 9. Complete RAG + Gemma function

RETRIEVAL_THRESHOLD = 0.50

source_names = {
    "labor_law": "نظام العمل",
    "executive_regulation": "اللائحة التنفيذية لنظام العمل",
    "workplace_regulation": "النموذج الموحد للائحة تنظيم العمل"
}


def ask_question_hybrid(question, k=3, threshold=RETRIEVAL_THRESHOLD):

    # ------------------------------------------------------------
    # 1. Retrieve relevant legal sources
    # ------------------------------------------------------------
    results = hybrid_search(question, k=k)

    best_score = results[0]["hybrid_score"]

    # ------------------------------------------------------------
    # 2. Prepare source information
    # ------------------------------------------------------------
    sources = []

    for rank, result in enumerate(results, start=1):

        chunk = rag_chunks[result["idx"]]
        metadata = chunk["metadata"]

        source_name = source_names.get(
            metadata["article_type"],
            metadata["article_type"]
        )

        sources.append({
            "rank": rank,
            "score": result["hybrid_score"],
            "article_type": metadata["article_type"],
            "source_name": source_name,
            "article": metadata["article"],
            "page": metadata["page_start"]
        })

    # ------------------------------------------------------------
    # 3. Reject weak retrieval
    # ------------------------------------------------------------
    if best_score < threshold:

        return {
            "answer": "لا يحتوي السياق المسترجع على معلومات كافية للإجابة.",
            "sources": sources,
            "rejected": True
        }

    # ------------------------------------------------------------
    # 4. Build legal context
    # ------------------------------------------------------------
    context_parts = []

    for rank, result in enumerate(results, start=1):

        chunk = rag_chunks[result["idx"]]
        metadata = chunk["metadata"]

        source_name = source_names.get(
            metadata["article_type"],
            metadata["article_type"]
        )

        context_parts.append(
            f"""المصدر {rank}
اسم المصدر: {source_name}
رقم المادة: {metadata["article"]}
الصفحة: {metadata["page_start"]}

النص القانوني:
{chunk["text"]}
"""
        )

    context = "\n\n".join(context_parts)

    # ------------------------------------------------------------
    # 5. Strict citation instructions
    # ------------------------------------------------------------
    user_prompt = f"""السؤال:
{question}

السياق القانوني المسترجع:
{context}

أجب عن السؤال اعتمادًا على النصوص الموجودة في السياق فقط.

قواعد إلزامية:
1. لا تستخدم أي معلومة من خارج السياق.
2. لا تخمّن أو تستنتج رقم مادة غير موجود في السياق.
3. إذا ذكرت رقم مادة، يجب أن يكون مطابقًا تمامًا لرقم المادة المكتوب في السياق.
4. إذا ذكرت اسم مصدر، يجب أن يكون مطابقًا تمامًا لاسم المصدر المكتوب في السياق.
5. لا تستخدم عبارات مثل "المصدر 1" أو "المصدر 2".
6. اذكر اسم المصدر ورقم المادة عند الاستشهاد بالحكم.
7. لا تخلط بين:
   - نظام العمل
   - اللائحة التنفيذية لنظام العمل
   - النموذج الموحد للائحة تنظيم العمل
8. إذا كانت المعلومات غير كافية للإجابة، قل:
   "لا يحتوي السياق المسترجع على معلومات كافية للإجابة."
9. أجب باللغة العربية فقط.
10. كن مختصرًا ومباشرًا.
"""

    # ------------------------------------------------------------
    # 6. Send to Gemma
    # ------------------------------------------------------------
    response = requests.post(
        "http://127.0.0.1:8080/v1/chat/completions",
        json={
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            "temperature": 0.0,
            "max_tokens": 300
        },
        timeout=120
    )

    response.raise_for_status()

    # ------------------------------------------------------------
    # 7. Extract answer
    # ------------------------------------------------------------
    answer = response.json()["choices"][0]["message"]["content"]

    # ------------------------------------------------------------
    # 8. Return result
    # ------------------------------------------------------------
    return {
        "answer": answer,
        "sources": sources,
        "rejected": False
    }

## 10. Full-system evaluation

These tests check three important behaviors:

1. A supported legal question.
2. An unsupported question that should be rejected.
3. Another supported legal question.


In [13]:
test_questions = [
    "ماذا يحدث لإجازة الوضع إذا أنجبت المرأة العاملة طفلًا مريضًا أو من ذوي الإعاقة ويحتاج إلى مرافق مستمر؟",
    "ما الحد الأقصى لمدة فترة التجربة؟",
    "ما هي مدة إجازة الوضع للمرأة العاملة إذا تأخر موعد الولادة عن التاريخ المرجح؟"
]

for i, question in enumerate(test_questions, start=1):

    print("=" * 70)
    print(f"TEST {i}")
    print(f"QUESTION: {question}")
    print("=" * 70)

    result = ask_question_hybrid(question)

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")
    for source in result["sources"]:
        print(
            f'Rank {source["rank"]} | '
            f'{source["article_type"]} | '
            f'المادة {source["article"]} | '
            f'الصفحة {source["page"]} | '
            f'score={source["score"]:.4f}'
        )

    print(f"\nRejected: {result['rejected']}")
    print()


TEST 1
QUESTION: ماذا يحدث لإجازة الوضع إذا أنجبت المرأة العاملة طفلًا مريضًا أو من ذوي الإعاقة ويحتاج إلى مرافق مستمر؟

ANSWER:
يحق للمرأة العاملة إجازة مدتها شهر بأجر كامل تبدأ بعد انتهاء مدة إجازة الوضع، في حالة إنجاب طفل مريض أو من ذوي الإعاقة ويتطلب حالته الصحية مرافقًا مستمرًا له. ولها الحق في تمديد الإجازة لمدة شهر دون أجر (النموذج الموحد للائحة تنظيم العمل، المادة 50).

SOURCES:
Rank 1 | workplace_regulation | المادة 50 | الصفحة 36 | score=0.5240
Rank 2 | executive_regulation | المادة 31 | الصفحة 22 | score=0.4895
Rank 3 | executive_regulation | المادة 9 | الصفحة 6 | score=0.4711

Rejected: False

TEST 2
QUESTION: ما الحد الأقصى لمدة فترة التجربة؟

ANSWER:
الحد الأقصى لمدة فترة التجربة هو 180 يومًا (المادة الثالثة والخمسون من نظام العمل).

SOURCES:
Rank 1 | workplace_regulation | المادة 8 | الصفحة 30 | score=0.5028
Rank 2 | labor_law | المادة الثالثة والخمسون | الصفحة 15 | score=0.4957
Rank 3 | executive_regulation | المادة 19 | الصفحة 15 | score=0.4731

Rejected: False

TEST 3